# Setup

## Import modules

In [ ]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from PIL import Image
import torch
from datetime import datetime

# Import pipeline modules
from scoring import score_image
from selection import select
from metrics import MetricsTracker
from generation.UnconditionalGenerator import UnconditionalGenerator
from generation.LatentSurvivalGenerator import LatentSurvivalGenerator
from utils import save_image, set_seed
import shutil
import random

# Config
OUTPUT_DIR = Path("outputs")

# Experiment parameters
NUM_PROMPTS = 5
NUM_TRIALS = 1
ROUNDS = 4        # r
BATCH_SIZE = 4    # B
SEED = 42
set_seed(SEED)

# Chosen strategies
SELECTION_STRATEGY = "argmax"

# Baseline scoring rubric (example)
# CLIP_RUBRIC = {"type": "clip", "text": "A photorealistic portrait of a dog", "weight": 1.0}
# BRIGHTNESS_RUBRIC = {"type": "brightness", "weight": 1.0}

## Load diffusion model

In [ ]:
from diffusers import AutoPipelineForText2Image

pipe = AutoPipelineForText2Image.from_pretrained("stabilityai/sdxl-turbo", torch_dtype=torch.float16, variant="fp16")
torch.cuda.empty_cache()
pipe.to("cuda")


## Load dataset

In [ ]:
# Load dev and test datasets 
dev_dataset = pd.read_csv("data/dev_dataset.csv")
test_dataset = pd.read_csv("data/test_dataset.csv")

# Run simulation

In [ ]:
START_IDX = 0
DATASET = dev_dataset[START_IDX : START_IDX + NUM_PROMPTS]

metrics = MetricsTracker()
# policy = UnconditionalGenerator(pipe, seed_safe=True)
policy = LatentSurvivalGenerator(pipe, alpha=1.0, gamma=1.0, seed_safe=True)

# based on date and time
RUN_DIR = OUTPUT_DIR / datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
RUN_DIR.mkdir(parents=True, exist_ok=True)

IMAGES_DIR = RUN_DIR / "images"
METRICS_CSV = RUN_DIR / "metrics.csv"

set_seed(SEED)

for trial in tqdm(range(NUM_TRIALS), desc="trials"):
    print("=" * 100)
    print(f"BEGINNING TRIAL {trial}")
    print("=" * 100)

    trial_id = f"trial{trial}"

    for item_idx, item in enumerate(tqdm(DATASET.to_dict(orient="records"), desc="prompts")):
        prompt_id = item["prompt_id"]
        prompt = item["prompt"]

        # Compile rubric
        rubric = {}
        for key, value in item.items():
            if key.startswith("rubric_"):
                rubric[key.replace("rubric_", "")] = value

        # Initialize policy to the new trial
        policy.initialize(prompt=prompt, batch_size=BATCH_SIZE)

        print(f"\nItem {item_idx}:")
        print(f"\tprompt_id={prompt_id}")
        print(f'\tprompt="{prompt}"')
        print(f"\trubric={rubric}")
        print()

        prompt_outdir = IMAGES_DIR / prompt_id / trial_id

        for round in range(ROUNDS):
            num_new_images = policy.num_new_images
            first_round = round == 0

            print(f"\tRound {round}")
            round_outdir = prompt_outdir / f"round{round}"
            round_outdir.mkdir(parents=True, exist_ok=True)

            # Score and record metrics for each image in the batch
            user_scores = []

            # If not the first round, include the previous winner as the first image in the batch
            if not first_round:
                image_path = round_outdir / f"image{0}.png"
                previous_winner = metrics.clone_previous_winner(prompt_id, trial_id, round, image_path)  # the current round  # the new image path

                user_scores.append(previous_winner["user_score"])
                # clone the previous winner image to the new destination
                shutil.copy(previous_winner["image_path"], image_path)

            # Generate the necessary number of images using our method
            seeds: list[int] = random.sample(range(1, 100001), num_new_images)
            new_images: list[Image.Image] = policy.generate(
                sampling_parameters={
                    "num_inference_steps": 1,
                    "guidance_scale": 0.0, # very important to set to 0.0 for SDXL-Turbo
                    "width": 512,
                    "height": 512,
                },
                seeds=seeds
            )

            for new_image_idx, new_image in enumerate(new_images):
                image_idx  = new_image_idx + (0 if first_round else 1)
                image_path = round_outdir / f"image{image_idx}.png"

                user_score = score_image(new_image, rubric)
                user_scores.append(user_score)
    
                metrics.log(
                    prompt_id,
                    trial_id,
                    round,
                    image_idx,
                    image_path,
                    user_score,
                    new_image,
                    extra_info={
                        "prompt": prompt,
                        "seed": seeds[new_image_idx],
                    },
                )
                save_image(new_image, image_path)

            # Select favorite (index)
            chosen_image_idx = select(user_scores, strategy=SELECTION_STRATEGY)
            metrics.mark_chosen(
                prompt_id,
                trial_id,
                round,
                chosen_image_idx,
            )

            # Update policy
            policy.update(winner_index=chosen_image_idx)

            print(f"\t\tImages saved to {round_outdir}")
            print(f"\t\tScores: {user_scores}")
            print(f"\t\tChosen index (using {SELECTION_STRATEGY}): {chosen_image_idx}")

        # Repeatedly save metrics per item, for safety
        metrics.save(str(METRICS_CSV))  # autoprints